# Bayes' Theorem: The Mathematical Formula

In the previous notebook we solved the disease-test problem by counting people. This notebook derives the **general formula** behind that computation, step by step, from the conditional-probability rules we already know. It is based on the eleventh video of Week 1 (Introduction to Probability).


## 1. Introduction

We want $P(\text{sick} \mid \text{diagnosed sick})$, and we know:

- $P(\text{sick}) = \frac{1}{10{,}000} = 0.0001$ — the key of the whole problem is that this is *tiny*: more people are misdiagnosed than are actually sick.
- $P(\text{healthy}) = 1 - 0.0001 = 0.9999$.
- The test is 99% effective for sick and healthy alike: $P(\text{diagnosed sick} \mid \text{sick}) = 0.99$ and $P(\text{diagnosed sick} \mid \text{healthy}) = 0.01$.

The goal is to assemble these four numbers into $P(\text{sick} \mid \text{diagnosed sick})$ using only rules from earlier notebooks.

After completing this notebook you will be able to:

- Derive Bayes' theorem from the definition of conditional probability.
- Expand $P(B)$ with the sum rule for disjoint events (the *law of total probability*).
- Apply the formula to any prior/evidence pair, not just the disease test.


## 2. Theory: the derivation

Let $A$ = sick and $B$ = diagnosed sick. We want $P(A \mid B)$.

### Step 1 — definition of conditional probability

From notebook 08, $P(A \mid B) \cdot P(B) = P(A \cap B)$. Dividing by $P(B)$:

$$
P(A \mid B) = \frac{P(A \cap B)}{P(B)}
$$

Now we need the numerator and the denominator.

### Step 2 — the numerator, by the product rule

$$
P(A \cap B) = P(A) \cdot P(B \mid A)
$$

— the probability of being sick times the probability of a correct diagnosis given sickness.

### Step 3 — the denominator, by the sum rule

The people diagnosed sick form the **union of two disjoint groups**: the sick who were correctly diagnosed, and the healthy who were misdiagnosed. You cannot be both sick and not sick, so the groups are disjoint and their probabilities add (notebook 04):

$$
P(B) = P(A \cap B) + P(A' \cap B)
$$

Applying the product rule to each term:

$$
P(B) = P(A)\,P(B \mid A) + P(A')\,P(B \mid A')
$$

This expansion is called the **law of total probability**.

### Step 4 — put it together: Bayes' theorem

$$
\boxed{\;P(A \mid B) = \frac{P(A)\,P(B \mid A)}{P(A)\,P(B \mid A) + P(A')\,P(B \mid A')}\;}
$$

Every piece came from a rule you already know: conditional probability, the product rule, and the sum rule for disjoint events.


## 3. Worked Example: plugging in the numbers

$$
P(\text{sick} \mid \text{diag. sick}) =
\frac{0.0001 \times 0.99}{0.0001 \times 0.99 + 0.9999 \times 0.01}
= \frac{0.000099}{0.000099 + 0.009999}
= \frac{0.000099}{0.010098}
\approx 0.0098
$$

Exactly the $\frac{99}{10{,}098} < 1\%$ we found by counting a million people — the formula is the counting argument, compressed.


## 4. Python Examples


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from fractions import Fraction

rng = np.random.default_rng(seed=42)


### The formula with exact fractions

Using `Fraction` shows the result is *exactly* 99/10,098 — the same integers as the counting argument.


In [ ]:
p_a = Fraction(1, 10_000)            # P(sick)
p_b_given_a = Fraction(99, 100)      # P(diagnosed sick | sick)
p_b_given_not_a = Fraction(1, 100)   # P(diagnosed sick | healthy)

numerator = p_a * p_b_given_a
denominator = numerator + (1 - p_a) * p_b_given_not_a
posterior = numerator / denominator

print(f"Numerator   P(A)P(B|A)           = {numerator}")
print(f"Denominator P(B), total prob.    = {denominator}")
print(f"P(A | B) = {posterior} = {float(posterior):.4f}")


### A general Bayes function

The same function works for any problem once you name the prior and the two likelihoods.


In [ ]:
def bayes(p_a, p_b_given_a, p_b_given_not_a):
    numerator = p_a * p_b_given_a
    return numerator / (numerator + (1 - p_a) * p_b_given_not_a)

# The disease test
print(f"Disease test:  {bayes(0.0001, 0.99, 0.01):.4f}")

# A different domain: spam filtering.
# 20% of email is spam; 'free' appears in 60% of spam, 4% of good mail.
print(f"Spam example:  P(spam | 'free') = {bayes(0.20, 0.60, 0.04):.4f}")


### Each term of the formula, checked by simulation

We simulate the population and measure all four ingredients plus the result — every quantity in the formula is observable.


In [ ]:
n = 5_000_000
is_sick = rng.random(n) < 0.0001
diagnosed_sick = np.where(is_sick, rng.random(n) < 0.99, rng.random(n) < 0.01)

print(f"P(A)          sim: {is_sick.mean():.5f}   theory: 0.00010")
print(f"P(B | A)      sim: {diagnosed_sick[is_sick].mean():.4f}    theory: 0.99")
print(f"P(B | A')     sim: {diagnosed_sick[~is_sick].mean():.4f}    theory: 0.01")
print(f"P(B)          sim: {diagnosed_sick.mean():.5f}   theory: 0.01010")
print(f"P(A | B)      sim: {is_sick[diagnosed_sick].mean():.4f}    theory: 0.0098")


## 5. Visualizations

### Mapping the tree onto the formula

The two shaded branches are the ones where $B$ (diagnosed sick) happens. The numerator of Bayes' theorem is the red branch; the denominator is red + orange.


In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 5.5))

root = (0, 2.5)
level1 = {"A (sick)": (3.5, 4.0), "A' (healthy)": (3.5, 1.0)}
level2 = {
    "A ∩ B": (7.5, 4.6), "A ∩ B'": (7.5, 3.4),
    "A' ∩ B": (7.5, 1.6), "A' ∩ B'": (7.5, 0.4),
}
formulas = {
    "A ∩ B": "P(A)·P(B|A) = 0.000099",
    "A ∩ B'": "",
    "A' ∩ B": "P(A')·P(B|A') = 0.009999",
    "A' ∩ B'": "",
}
colors = {"A ∩ B": "#e15759", "A' ∩ B": "#f28e2b",
          "A ∩ B'": "#cccccc", "A' ∩ B'": "#cccccc"}

def branch(a, b, label):
    ax.plot([a[0], b[0]], [a[1], b[1]], color="gray", linewidth=1.5)
    mx, my = (a[0] + b[0]) / 2, (a[1] + b[1]) / 2
    ax.annotate(label, (mx, my), textcoords="offset points", xytext=(0, 6),
                ha="center", fontsize=9, color="darkblue")

branch(root, level1["A (sick)"], "P(A) = 0.0001")
branch(root, level1["A' (healthy)"], "P(A') = 0.9999")
branch(level1["A (sick)"], level2["A ∩ B"], "P(B|A) = 0.99")
branch(level1["A (sick)"], level2["A ∩ B'"], "0.01")
branch(level1["A' (healthy)"], level2["A' ∩ B"], "P(B|A') = 0.01")
branch(level1["A' (healthy)"], level2["A' ∩ B'"], "0.99")

ax.plot(*root, "o", markersize=9, color="black")
for name, pos in level1.items():
    ax.plot(*pos, "o", markersize=8, color="#4e79a7")
    ax.annotate(name, pos, textcoords="offset points", xytext=(0, 9),
                ha="center", fontsize=10)
for name, pos in level2.items():
    ax.plot(*pos, "s", markersize=9, color=colors[name])
    label = name if not formulas[name] else f"{name}:  {formulas[name]}"
    ax.annotate(label, pos, textcoords="offset points", xytext=(10, 0),
                ha="left", fontsize=10,
                fontweight="bold" if formulas[name] else "normal")

ax.text(4.5, -0.9,
        "P(A | B) = red / (red + orange) = 0.000099 / 0.010098 ≈ 0.0098",
        fontsize=12, ha="center",
        bbox=dict(boxstyle="round", facecolor="#fff3cd"))

ax.set_xlim(-1.5, 11.5)
ax.set_ylim(-1.4, 5.3)
ax.axis("off")
ax.set_title("Bayes' theorem is the tree: one branch over the sum of the B-branches")
plt.show()


### Posterior as a function of test accuracy and prevalence

The formula makes it cheap to explore: each curve is one test accuracy; the x-axis sweeps how rare the disease is.


In [ ]:
prevalences = np.logspace(-5, -1, 200)

plt.figure(figsize=(9, 5))
for eff in [0.9, 0.99, 0.999, 0.9999]:
    posteriors = [bayes(p, eff, 1 - eff) for p in prevalences]
    plt.semilogx(prevalences, posteriors, label=f"{eff:.2%} effective test")

plt.axvline(1e-4, color="gray", linestyle=":", label="our example (1 in 10,000)")
plt.title("P(sick | positive): the prior matters as much as the test")
plt.xlabel("P(sick) — prevalence (log scale)")
plt.ylabel("P(sick | positive)")
plt.grid(True, alpha=0.3, which="both")
plt.legend()
plt.show()


## 6. Exercises

### Basic

1. Identify $P(A)$, $P(B \mid A)$, and $P(B \mid A')$ in this problem: 5% of products are defective; a quality check flags 98% of defective products and 3% of good ones. A product got flagged.
2. Compute $P(\text{defective} \mid \text{flagged})$ with the `bayes` function or by hand.

### Intermediate

3. Write the formula for $P(A' \mid B')$ — the probability of being healthy given a *negative* test — and evaluate it for the disease example.
4. In the spam example ($P(\text{spam}) = 0.2$, $P(\text{'free'} \mid \text{spam}) = 0.6$, $P(\text{'free'} \mid \text{ham}) = 0.04$), what does the posterior become if 'free' appears in 20% of good mail instead of 4%?

### Advanced

5. **Monty Hall** (the video announces a lab on it): you pick door 1 of 3; the host, who knows where the car is, opens a goat door among doors 2-3; you may switch. Use Bayes' theorem to compute $P(\text{car behind your door} \mid \text{host's action})$, then verify by simulating 100,000 games that switching wins 2/3 of the time.


## 7. Solutions


In [ ]:
# Exercises 1-2: defective products
p = bayes(0.05, 0.98, 0.03)
print(f"1-2.  P(A)=0.05, P(B|A)=0.98, P(B|A')=0.03 -> "
      f"P(defective | flagged) = {p:.4f}")

# Exercise 3: healthy given negative test — same formula with roles swapped:
# P(A'|B') = P(A')P(B'|A') / (P(A')P(B'|A') + P(A)P(B'|A))
p_healthy_given_neg = bayes(0.9999, 0.99, 0.01)
print(f"3.    P(healthy | negative) = {p_healthy_given_neg:.8f}")

# Exercise 4: noisier keyword
print(f"4.    P(spam | 'free') drops to {bayes(0.20, 0.60, 0.20):.4f}")

# Exercise 5: Monty Hall.
# Prior: P(car behind door 1) = 1/3. Host opens door 3 (goat).
# P(host opens 3 | car at 1) = 1/2, P(host opens 3 | car at 2) = 1,
# P(host opens 3 | car at 3) = 0. Bayes:
p_stay = (Fraction(1, 3) * Fraction(1, 2)) / (
    Fraction(1, 3) * Fraction(1, 2) + Fraction(1, 3) * 1 + Fraction(1, 3) * 0)
print(f"5.    P(car at your door | host opened 3) = {p_stay} -> switching wins "
      f"{1 - p_stay}")

n_games = 100_000
car = rng.integers(0, 3, size=n_games)
pick = np.zeros(n_games, dtype=int)          # always pick door 0
stay_wins = (car == pick).mean()
print(f"      Simulated: staying wins {stay_wins:.3f}, "
      f"switching wins {1 - stay_wins:.3f}")


## 8. Conclusion

- Bayes' theorem is assembled from three known rules: **conditional probability** ($P(A \mid B) = P(A \cap B)/P(B)$), the **product rule** (numerator), and the **sum rule for disjoint events** expanding $P(B)$ (the law of total probability).

$$
P(A \mid B) = \frac{P(A)\,P(B \mid A)}{P(A)\,P(B \mid A) + P(A')\,P(B \mid A')}
$$

- Plugging in the disease numbers reproduces the counting result exactly: $99/10{,}098 \approx 0.98\%$.
- The formula transfers unchanged to any domain — quality control, spam filtering, Monty Hall — once you identify the prior $P(A)$ and the likelihoods $P(B \mid A)$, $P(B \mid A')$.
